<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/06_live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · 6. Live session

Runs the whole live loop on a free Colab T4 for a few hours: vLLM serves the fine-tuned model, and the watcher
polls SEC EDGAR every 30 seconds for new earnings releases, answers each one and logs the answer, the SEC
acceptance time and how long it took. Free Colab sessions end after a few hours (sooner if idle), so this is for
busy earnings mornings and afternoons, not 24/7; the scheduled GitHub job in `docs/LIVE.md` is the always-on
option. Use a **fresh** runtime: vLLM installs its own torch.

Releases are filed mostly between 6 and 9 am and 4 and 6 pm Eastern on weekdays.

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"   # to try a branch before merging it, put its name here
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements-serve.txt

In [ ]:
# Keep data, checkpoints and results on Google Drive so a disconnect doesn't lose them
# and every notebook sees the same dataset.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!mkdir -p /content/drive/MyDrive/finsight/data
!rm -rf outputs results data && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results && ln -s /content/drive/MyDrive/finsight/data data

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
# The SEC asks every client to identify itself with a name and email (sent only to sec.gov).
# Store it once as a Colab secret named SEC_USER_AGENT, or type it when asked.
import os
try:
    from google.colab import userdata
    os.environ["SEC_USER_AGENT"] = userdata.get("SEC_USER_AGENT")
except Exception:
    os.environ["SEC_USER_AGENT"] = input("Your name and email for the SEC, e.g. Jane Doe jane@example.com: ")

In [ ]:
import subprocess, time, urllib.request
vllm = subprocess.Popen(["bash", "scripts/serve_vllm.sh", "outputs/finsight-earnings/final"],
                        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health"); print("vLLM is up"); break
    except Exception:
        time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])

## Watch
Stop the cell whenever you like; everything already answered is saved on Drive. The first poll is marked `catch_up` (it picks up recent filings from before you started), so it does not count toward detection speed.

In [ ]:
LIVE = "/content/drive/MyDrive/finsight/live"
!python -m finsight.live --live-dir $LIVE poll --model-url http://localhost:8000 --loop --interval 30 --max-hours 5

## Score earlier answers and build the page
Scoring only finds figures for releases from at least a few days ago, once their 10-Q is out.

In [ ]:
!python -m finsight.live --live-dir $LIVE score
!python -m finsight.dashboard --live $LIVE --out site
from IPython.display import HTML
HTML(open("site/index.html").read())

## Optional: add this session to the public log
Merges the session into the repo's `live-data` branch, which the public page is built from. Needs a GitHub fine-grained token with *Contents: read and write* on this repo only, stored as the Colab secret `GH_TOKEN`.

In [ ]:
import os, subprocess
from google.colab import userdata
url = "https://x-access-token:" + userdata.get("GH_TOKEN") + "@github.com/PujaSridhar/finsight"
subprocess.run(["rm", "-rf", "/content/live-data"])
if subprocess.run(["git", "clone", "-q", "-b", "live-data", url, "/content/live-data"]).returncode:
    print("No live-data branch yet: run the live workflow once from the Actions tab (task: page), then re-run this cell.")
else:
    subprocess.run(["python", "-m", "finsight.live", "--live-dir", "/content/live-data", "merge", LIVE], check=True)
    git = ["git", "-C", "/content/live-data", "-c", "user.name=FinSight live", "-c", "user.email=finsight-live@localhost"]
    subprocess.run(git + ["add", "-A"], check=True)
    if subprocess.run(git + ["diff", "--cached", "--quiet"]).returncode:
        subprocess.run(git + ["commit", "-qm", "live: Colab session"], check=True)
        subprocess.run(git + ["push", "-q"], check=True)
        print("pushed; the page updates on the next hourly run")
    else:
        print("nothing new to add")